<a href="https://colab.research.google.com/github/Fazilsobhani/FazilSobhani/blob/main/TinyMLFazilJune2026F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import math
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
import os

print("TensorFlow Version:", tf.__version__)

# ==========================================
# 1. تولید داده و آموزش مدل تنسورفلو
# ==========================================
print("\n--- 1. Generating Data and Training Model ---")
SAMPLES = 2000
np.random.seed(42)

x_values = np.random.uniform(low=0, high=2*math.pi, size=SAMPLES).astype(np.float32)
np.random.shuffle(x_values)
y_values = np.sin(x_values) + (0.1 * np.random.randn(*x_values.shape)).astype(np.float32)

x_train = x_values[:int(0.8 * SAMPLES)].reshape(-1, 1)
y_train = y_values[:int(0.8 * SAMPLES)].reshape(-1, 1)

# ساخت و آموزش شبکه
tf_model = models.Sequential([
    layers.Dense(16, activation='relu', input_shape=(1,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])
tf_model.compile(optimizer='adam', loss='mse')
tf_model.fit(x_train, y_train, epochs=400, batch_size=16, verbose=0)
print("Training Completed!")

# ==========================================
# 2. کوانتیزاسیون و تبدیل به TFLite
# ==========================================
print("\n--- 2. Quantizing Model to INT8 ---")
converter = tf.lite.TFLiteConverter.from_keras_model(tf_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset_gen():
    for _ in range(100):
        data = np.random.uniform(low=0, high=2*np.pi, size=(1, 1)).astype(np.float32)
        yield [data]

converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

tflite_model_name = "sine_model_quantized.tflite"
with open(tflite_model_name, "wb") as f:
    f.write(tflite_quant_model)
print(f"Size of Quantized Model: {os.path.getsize(tflite_model_name)} bytes")

# ==========================================
# 3. تبدیل به آرایه C++
# ==========================================
print("\n--- 3. Generating C++ Header File ---")
def convert_to_c_array(bytes_data, array_name="g_model"):
    hex_array = [hex(b) for b in bytes_data]
    c_str = f"const unsigned char {array_name}[] = {{\n"
    for i in range(0, len(hex_array), 12):
        c_str += "    " + ", ".join(hex_array[i:i+12]) + ",\n"
    c_str += "};\n"
    c_str += f"const int {array_name}_len = {len(hex_array)};\n"
    return c_str

c_code = convert_to_c_array(tflite_quant_model, "sine_model_quantized_tflite")

cc_file_name = "sine_model_data.cc"
with open(cc_file_name, "w") as f:
    f.write(c_code)
print(f"Success! C++ Array saved as '{cc_file_name}'")

TensorFlow Version: 2.20.0

--- 1. Generating Data and Training Model ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Completed!

--- 2. Quantizing Model to INT8 ---
Saved artifact at '/tmp/tmpig_7avp9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140010983617936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140010983616016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140010983617744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140010983616400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140010983619664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140010983616592: TensorSpec(shape=(), dtype=tf.resource, name=None)
Size of Quantized Model: 3344 bytes

--- 3. Generating C++ Header File ---
Success! C++ Array saved as 'sine_model_data.cc'


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
